In [3]:
"""
05_algorithmic_fairness.py

Evaluates the algorithmic fairness of the 26-gene clinical XGBoost model.
Stratifies predictive performance (AUROC) across protected demographic 
attributes (Sex and Age) by dynamically linking the raw clinical metadata.
"""

import warnings
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import xgboost as xgb
from sklearn.metrics import roc_curve, auc

warnings.filterwarnings("ignore")

# ==========================================
# CONFIGURATION
# ==========================================
BASE_DIR = Path.cwd().parent if "notebooks" in str(Path.cwd()) else Path(__file__).resolve().parents[2]
DATA_DIR = BASE_DIR / "data" / "processed" / "ml_tensors"
DEG_DIR = BASE_DIR / "data" / "processed" / "deg_tensors"
RAW_META_DIR = BASE_DIR / "data" / "raw" / "geo_metadata"
PLOT_DATA_DIR = BASE_DIR / "outputs" / "plot_data"
MODEL_DIR = BASE_DIR / "outputs" / "models"
FIG_OUT = BASE_DIR / "outputs" / "figures"

HOLDOUT_COHORT = 'GSE65682'

class SklearnCompatibleXGBClassifier(xgb.XGBClassifier):
    def __sklearn_tags__(self):
        tags = super().__sklearn_tags__()
        if hasattr(tags, "estimator_type"):
            tags.estimator_type = "classifier"
        elif isinstance(tags, dict):
            tags["estimator_type"] = "classifier"
        return tags

def calculate_subgroup_auc(y_true, y_probs):
    """Calculates AUROC and handles edge cases where only one class is present."""
    if len(np.unique(y_true)) < 2:
        return np.nan
    fpr, tpr, _ = roc_curve(y_true, y_probs)
    return auc(fpr, tpr)

def main():
    print("[*] INITIATING ALGORITHMIC FAIRNESS EVALUATION...")
    FIG_OUT.mkdir(parents=True, exist_ok=True)

    # 1. Load Data
    print("    -> Loading Tensors and extracting raw Demographics...")
    features_path = PLOT_DATA_DIR / "03_optimal_feature_list.csv"
    optimal_genes = pd.read_csv(features_path)['Optimal_Genes'].tolist()

    X_elite = pd.read_csv(DEG_DIR / "X_deg_master.csv.gz", compression='gzip')
    y = pd.read_csv(DATA_DIR / "y_master.csv")
    meta = pd.read_csv(DATA_DIR / "meta_master.csv")
    
    # 2. Extract Vault Cohort
    test_mask = meta['Dataset'] == HOLDOUT_COHORT
    X_vault = X_elite.loc[test_mask, optimal_genes]
    
    target_col = 'Mortality' if 'Mortality' in y.columns else y.columns[0]
    
    # [THE FIX]: Strip the old index so it aligns perfectly with the merged metadata
    y_vault = y.loc[test_mask, target_col].astype(int).reset_index(drop=True)
    
    meta_vault = meta.loc[test_mask].copy()

    # Load exact Age and Sex from raw GEO metadata
    vault_raw_meta = pd.read_csv(RAW_META_DIR / f"{HOLDOUT_COHORT}_metadata.csv")
    meta_vault = pd.merge(meta_vault, vault_raw_meta, on='Patient_ID', how='left')

    # Clean Sex
    if 'gender' in meta_vault.columns:
        meta_vault['Clean_Sex'] = meta_vault['gender'].astype(str).str.strip().str.title()
        meta_vault['Clean_Sex'] = meta_vault['Clean_Sex'].replace({'M': 'Male', 'F': 'Female'})
    else:
        meta_vault['Clean_Sex'] = 'Unknown'

    # Clean Age
    if 'age' in meta_vault.columns:
        meta_vault['Clean_Age'] = pd.to_numeric(meta_vault['age'], errors='coerce')
        conditions = [(meta_vault['Clean_Age'] < 60), (meta_vault['Clean_Age'] >= 60)]
        choices = ['< 60 Years', '>= 60 Years']
        meta_vault['Age_Group'] = np.select(conditions, choices, default='Unknown')
    else:
        meta_vault['Age_Group'] = 'Unknown'

    # 3. Load Model and Predict
    model_path = MODEL_DIR / "sepsis_xgboost_26genes_clinical_v1.json"
    print(f"    -> Loading locked weights from {model_path.name}...")
    model = xgb.XGBClassifier()
    model.load_model(model_path)
    
    vault_probs = model.predict_proba(X_vault)[:, 1]
    
    # 4. Calculate Subgroup Metrics
    print("\n[*] FAIRNESS METRICS (AUROC):")
    results = []

    # Overall baseline
    baseline_auc = calculate_subgroup_auc(y_vault, vault_probs)
    results.append({'Category': 'Baseline', 'Subgroup': 'Overall Cohort', 'AUROC': baseline_auc, 'N': len(y_vault)})
    print(f"    -> Overall Vault AUROC : {baseline_auc:.3f}")

    # Sex Subgroups
    for sex in ['Male', 'Female']:
        # [THE FIX]: Convert mask to raw numpy array using .values to prevent indexing errors
        mask = (meta_vault['Clean_Sex'] == sex).values
        if mask.sum() > 0:
            val = calculate_subgroup_auc(y_vault[mask], vault_probs[mask])
            results.append({'Category': 'Biological Sex', 'Subgroup': sex, 'AUROC': val, 'N': mask.sum()})
            print(f"    -> Sex: {sex:<6} (N={mask.sum():<3}) : {val:.3f}")

    # Age Subgroups
    for age_grp in ['< 60 Years', '>= 60 Years']:
        # [THE FIX]: Convert mask to raw numpy array
        mask = (meta_vault['Age_Group'] == age_grp).values
        if mask.sum() > 0:
            val = calculate_subgroup_auc(y_vault[mask], vault_probs[mask])
            results.append({'Category': 'Age Bracket', 'Subgroup': age_grp, 'AUROC': val, 'N': mask.sum()})
            print(f"    -> Age: {age_grp:<10} (N={mask.sum():<3}) : {val:.3f}")

    df_results = pd.DataFrame(results).dropna()

    if len(df_results) == 1:
        print("\n[!] WARNING: Subgroups still missing. Check if GSE65682_metadata.csv contains 'age' and 'gender' columns.")
        return

    # 5. Plotting
    print("\n    -> Generating Fairness Bar Chart...")
    plt.figure(figsize=(10, 6))
    sns.set_theme(style="whitegrid", rc={"axes.edgecolor": "#aaaaaa", "grid.color": "#eeeeee"})
    
    ax = sns.barplot(
        data=df_results, 
        y='Subgroup', 
        x='AUROC', 
        hue='Category',
        dodge=False,
        palette={"Baseline": "#333333", "Biological Sex": "#4a6fe3", "Age Bracket": "#db4325"}
    )

    plt.axvline(x=baseline_auc, color='#333333', linestyle='--', linewidth=1.5, alpha=0.7, zorder=0)
    plt.text(baseline_auc + 0.01, -0.4, f'Baseline: {baseline_auc:.3f}', color='#333333', fontsize=10)

    for i, p in enumerate(ax.patches):
        width = p.get_width()
        if width > 0:
            n_val = df_results.iloc[i]['N']
            ax.text(width - 0.08, p.get_y() + p.get_height()/2.,
                    f'{width:.3f} (n={n_val})', 
                    va='center', color='white', fontweight='bold', fontsize=10)

    plt.xlim(0, 1.0)
    plt.title('Algorithmic Fairness: Demographic Performance Parity', fontsize=14, pad=15, color='#333333')
    plt.xlabel('Area Under the ROC Curve (AUROC)', fontsize=12, labelpad=10, color='#555555')
    plt.ylabel('')
    
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', frameon=False)
    
    plt.tight_layout()
    pdf_out = FIG_OUT / "Fig10_Algorithmic_Fairness.pdf"
    plt.savefig(pdf_out, dpi=300, bbox_inches='tight')
    plt.close()

    print(f"[*] SUCCESS! Plot saved to {pdf_out.name}")

if __name__ == "__main__":
    main()

[*] INITIATING ALGORITHMIC FAIRNESS EVALUATION...
    -> Loading Tensors and extracting raw Demographics...
    -> Loading locked weights from sepsis_xgboost_26genes_clinical_v1.json...

[*] FAIRNESS METRICS (AUROC):
    -> Overall Vault AUROC : 0.635
    -> Sex: Male   (N=272) : 0.633
    -> Sex: Female (N=207) : 0.638
    -> Age: < 60 Years (N=183) : 0.659
    -> Age: >= 60 Years (N=296) : 0.622

    -> Generating Fairness Bar Chart...
[*] SUCCESS! Plot saved to Fig10_Algorithmic_Fairness.pdf
